# Etapa 4 — Retrieval Semántico y Representaciones de Contenido Textual
## Cuaderno de Reproducibilidad para Google Colab

Este cuaderno permite reproducir y verificar de forma liviana todo el pipeline experimental de la **Etapa 4** de la tesis:
- Integración de embeddings de texto preentrenados (`all-MiniLM-L6-v2`) en la torre de ítems del Two-Tower.
- Comparación de 9 variantes arquitectónicas (**A, B, C, C-cero, C2, D, D2, E, R**).
- Evaluación en dos escenarios: **Estándar** (catálogo completo) y **Cold-Start** (10% de ítems estrictamente fríos sin interacciones en train).
- Análisis de sesgo de popularidad y generación de candidatos en dos etapas.

---
### Leyenda de Variantes Evaluadas
Tomada de las especificaciones metodológicas del proyecto:
- **A**: Two-Tower base solo por ID de usuario e ítem (reproduce exactamente el retrieval de la Etapa 3).
- **B**: Two-Tower con embedding de texto preentrenado (`all-MiniLM-L6-v2`) proyectado por MLP ($384 \rightarrow 64 \rightarrow 32$), sin embedding por ID de ítem.
- **C**: Concatenación de embedding de ID (32) + proyección de texto (32), total 64 dim.
- **C-cero**: Variante C con pesos entrenados congelados, anulando en inferencia el embedding por ID para ítems fríos (control de inferencia).
- **C2**: Variante C con ID Dropout ($p=0.5$ por muestra en entrenamiento) e ID anulado en inferencia para ítems fríos.
- **D**: Concatenación de embedding de ID (32) + proyección de géneros ($19 \rightarrow 64 \rightarrow 32$), total 64 dim.
- **D2**: Variante D con ID Dropout ($p=0.5$ por muestra en entrenamiento) e ID anulado en inferencia para ítems fríos (control de C2 con géneros).
- **E**: Solo embedding de usuario + proyección de géneros ($19 \rightarrow 64 \rightarrow 32$), sin ID (control de B con géneros).
- **R**: Control de contenido con embedding aleatorio fijo (384 dim normalizado) + proyección MLP ($384 \rightarrow 64 \rightarrow 32$), sin ID.

---
> **NOTA VISIBLE METODOLÓGICA Y LIMITACIONES**:
> 1. **Entorno de Referencia**: Las tablas de referencia oficiales de este proyecto fueron registradas en la GPU local del autor (**NVIDIA GeForce RTX 4060 Laptop 8GB**, PyTorch `2.6.0+cu124`, CUDA 12.4, Python 3.11, Windows 11).
> 2. **Variabilidad en Colab**: Las ejecuciones en Google Colab pueden presentar leves discrepancias numéricas respecto a la máquina local debido a la GPU asignada (ej. T4, L4, A100), los controladores y kernels CUDA de Linux. Esto incide principalmente en las métricas de Two-Stage (que dependen del entrenamiento del ranker MLP); el Retrieval puro coincide estrechamente.
> 3. **Split en Frío Idéntico**: En el escenario en frío (`cold_start`), los ítems fríos (972 ítems, 10% del catálogo) son exactamente los mismos para todas las semillas (partición determinista con `split_seed=42`).
> 4. **Tiempo Estimado**: En la GPU local (RTX 4060), cada semilla completa insume aproximadamente **12 minutos** para entrenar y evaluar las 9 variantes en ambos escenarios. En Google Colab el tiempo exacto por semilla dependerá del hardware que asigne la plataforma.

--- 
## 1. Descarga del proyecto y posicionamiento en el directorio
Configura la rama/etiqueta a clonar y las semillas a ejecutar. Por defecto clona la etiqueta `etapa4-final` (o la rama activa `04-semantic-retrieval`).

In [ ]:
# @title Configuración de descarga del proyecto
GIT_REF = "etapa4-final"  #@param {type:"string"}
METODO = "github"  #@param ["github", "upload_zip", "drive_zip"]
REPO_URL = "https://github.com/marcosgoizueta/tesis-03-two-stage.git"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/03-two-stage.zip"  #@param {type:"string"}

# Configuración de semillas a ejecutar:
# Por defecto solo semilla 42. Para 3 semillas: [42, 43, 44]. Para las 10 completas: list(range(42, 52))
SEEDS = [42]  #@param {type:"raw"}

# Opción para persistir resultados en Google Drive (permite usar --resume si se corta la sesión)
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/recsys_stage4_results"  #@param {type:"string"}

import os, sys, glob, subprocess

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    print(f"Directorio en Drive activo: {DRIVE_RESULTS_DIR}")

if METODO == "github":
    repo_name = os.path.basename(REPO_URL).replace('.git', '')
    if not os.path.exists(f"/content/{repo_name}"):
        print(f"Clonando {REPO_URL}...")
        !git clone {REPO_URL} /content/{repo_name}
    %cd /content/{repo_name}
    print(f"Haciendo checkout a la referencia '{GIT_REF}'...")
    checkout_res = subprocess.run(["git", "checkout", GIT_REF], capture_output=True, text=True)
    if checkout_res.returncode != 0:
        print("\n" + "!" * 75)
        print(f"⚠️ ERROR: No se pudo hacer checkout a la referencia '{GIT_REF}'.")
        print("Detalle de Git:", checkout_res.stderr.strip() or checkout_res.stdout.strip())
        print("Por favor, verificá que la etiqueta o rama exista en el repositorio remoto.")
        print("Podés cambiar el valor de GIT_REF en el formulario superior (ej. '04-semantic-retrieval' o un commit hash).")
        print("!" * 75 + "\n")
        raise RuntimeError(f"Referencia de Git '{GIT_REF}' no encontrada en el repositorio. Verificá la rama o etiqueta.")
    else:
        print(f"✅ Checkout exitoso a '{GIT_REF}'.")

elif METODO == "upload_zip":
    zips = glob.glob("/content/*.zip")
    if not zips:
        raise FileNotFoundError("⚠️ Arrastrá el archivo .zip al panel de archivos de Colab y volvé a correr esta celda.")
    zip_file = zips[0]
    print(f"Descomprimiendo {zip_file}...")
    !unzip -q -o {zip_file} -d /content/03-two-stage
    %cd /content/03-two-stage

elif METODO == "drive_zip":
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    !unzip -q -o {DRIVE_ZIP_PATH} -d /content/03-two-stage
    %cd /content/03-two-stage

# Asegurar que estamos en el directorio con requirements-colab-stage4.txt
if not os.path.exists("requirements-colab-stage4.txt"):
    subdirs = [d for d in glob.glob("**/requirements-colab-stage4.txt", recursive=True)]
    if subdirs:
        target_dir = os.path.dirname(subdirs[0])
        %cd {target_dir}

print("\nDirectorio de trabajo actual:", os.getcwd())
print("Archivos principales encontrados:", [f for f in os.listdir('.') if not f.startswith('.')])

--- 
## 2. Instalación de dependencias y verificación del entorno (GPU y Librerías)
Instala las dependencias específicas de la Etapa 4 (`requirements-colab-stage4.txt`, sin fijar `torch` ni el índice de CUDA para respetar el entorno nativo de Colab) y compara las versiones con las registradas en el directorio de referencia `reference/stage4/metadata.json`.

In [ ]:
# Instalación de dependencias para Etapa 4 en Colab
if os.path.exists("requirements-colab-stage4.txt"):
    !pip install -q -r requirements-colab-stage4.txt
else:
    !pip install -q sentence-transformers==3.0.1 transformers==4.57.6 scikit-learn pandas numpy matplotlib pytest

import json, torch
import sentence_transformers, transformers

print("=" * 70)
print("VERIFICACIÓN DE ACELERADOR GPU Y LIBRERÍAS")
print("=" * 70)

if not torch.cuda.is_available():
    print("\n" + "!" * 70)
    print("⚠️ ADVERTENCIA CRÍTICA: NO SE DETECTÓ GPU ACTIVA EN GOOGLE COLAB.")
    print("La Etapa 4 entrena 7 variantes Two-Tower y evalúa candidatos en dos escenarios;")
    print("en CPU la corrida sería extremadamente lenta (~horas por semilla).")
    print("Por favor, activá la GPU en: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU.")
    print("!" * 70 + "\n")
else:
    print(f"GPU detectada    : {torch.cuda.get_device_name(0)}")
    print(f"CUDA en PyTorch  : {torch.version.cuda}")
    print(f"Capacidad Compute: {torch.cuda.get_device_capability(0)}")
    !nvidia-smi

print("\n--- Comparación de Librerías vs Referencia Local (reference/stage4/metadata.json) ---")
active_versions = {
    "torch": torch.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "transformers": transformers.__version__,
}

ref_versions = {}
ref_meta_path = "reference/stage4/metadata.json" if os.path.exists("reference/stage4/metadata.json") else "data/cache/metadata.json"
if os.path.exists(ref_meta_path):
    with open(ref_meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
        ref_versions = meta.get("libraries", {})

for lib, act_v in active_versions.items():
    ref_v = ref_versions.get(lib, "(no registrado en metadata.json)")
    match_str = "✅ Coincide" if act_v == ref_v else "ℹ️ Difiere (esperable en Colab)"
    print(f"{lib:22}: Activa = {act_v:18} | Referencia = {ref_v:18} -> {match_str}")
print("=" * 70)

--- 
## 3. Generación y Validación de Embeddings de Texto (`all-MiniLM-L6-v2`)
Por defecto, los embeddings de las 9.724 películas se generan en Colab usando la plantilla `<title>. Genres: <genres>` y se guardan en `data/cache/`. Si existe la matriz de referencia en `reference/stage4/movie_embeddings_minilm.npy`, calcula la similitud coseno media y la discrepancia máxima para auditar la consistencia numérica entre entornos. Si falta la matriz de referencia, el notebook continúa normalmente.

In [ ]:
# @title Validación de matriz de embeddings
REFERENCE_EMBEDDINGS_PATH = "reference/stage4/movie_embeddings_minilm.npy"  #@param {type:"string"}

import numpy as np
sys.path.insert(0, "src")
from recsys.data import load_and_prepare_data, ensure_dataset
from recsys.stage4.embeddings import generate_or_load_item_embeddings
from pathlib import Path

ensure_dataset(Path("data"))
_, _, movies_df, _ = load_and_prepare_data(Path("data"))

# Carga o genera la matriz en Colab dentro de data/cache/
embs_colab, meta_colab = generate_or_load_item_embeddings(
    movies_df=movies_df,
    cache_dir=Path("data/cache"),
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print(f"Matriz generada/cargada en Colab: shape={embs_colab.shape}, dtype={embs_colab.dtype}")

# Si existe matriz de referencia independiente, contrastamos numéricamente
if os.path.exists(REFERENCE_EMBEDDINGS_PATH):
    embs_ref = np.load(REFERENCE_EMBEDDINGS_PATH)
    print(f"Matriz de referencia cargada desde '{REFERENCE_EMBEDDINGS_PATH}': shape={embs_ref.shape}")
    
    # Similitud coseno por ítem (asumiendo vectores normalizados)
    cos_sims = np.sum(embs_colab * embs_ref, axis=1)
    max_abs_diff = np.max(np.abs(embs_colab - embs_ref))
    
    print(f"Similitud coseno media : {np.mean(cos_sims):.8f}")
    print(f"Similitud coseno mínima: {np.min(cos_sims):.8f}")
    print(f"Diferencia absoluta máx: {max_abs_diff:.8e}")
    if np.mean(cos_sims) > 0.99999:
        print("✅ La matriz generada en Colab es numéricamente idéntica a la referencia local.")
    else:
        print("ℹ️ Leves fluctuaciones por diferencias de arquitectura/kernels CUDA en Colab.")
else:
    print(f"ℹ️ Archivo de referencia '{REFERENCE_EMBEDDINGS_PATH}' no encontrado (no fue subido al repo).")
    print("El notebook continúa utilizando normalmente la matriz recién generada en Colab.")

--- 
## 4. Validación Rápida del Pipeline (`--quick`)
Ejecuta una corrida corta de prueba (1 época y subconjunto acotado de usuarios) para todas las variantes y escenarios, comprobando que todo el flujo funcione en pocos segundos en Colab.

In [ ]:
!python scripts/run_stage4.py --quick --device auto --guard warn --results-dir results/colab/quick

--- 
## 5. Ejecución Completa con Semillas Configuradas
Ejecuta las variantes para las semillas elegidas en `SEEDS` (por defecto `[42]`). Mide el tiempo de ejecución y usa `--guard warn` para comprobar la Variante A.

In [ ]:
import time

seeds_str = " ".join(map(str, SEEDS))
target_results_dir = DRIVE_RESULTS_DIR if SAVE_TO_DRIVE else "results/colab"

print(f"Iniciando corrida completa de Etapa 4 para {len(SEEDS)} semilla(s): {SEEDS}")
print(f"Directorio de resultados: {target_results_dir}")
print(f"Guardia de Variante A  : warn")

t0 = time.time()
!python scripts/run_stage4.py --seeds {seeds_str} --device auto --guard warn --results-dir {target_results_dir} --resume
elapsed = time.time() - t0

mins = int(elapsed // 60)
secs = int(elapsed % 60)
print(f"\nCorrida finalizada con éxito. Tiempo total: {mins}m {secs}s ({elapsed:.2f} s)")

--- 
## 6. Visualización de Resultados Obtenidos en Colab
Muestra el resumen de resultados de la corrida actual en Colab y, si se ejecutaron múltiples semillas, los contrastes pareados primarios.

In [ ]:
import pandas as pd
import glob

target_results_dir = DRIVE_RESULTS_DIR if SAVE_TO_DRIVE else "results/colab"
summary_files = glob.glob(f"{target_results_dir}/*summary*.csv") + glob.glob(f"{target_results_dir}/stage4_results_*.csv")

if summary_files:
    latest_res = sorted(summary_files)[-1]
    df_colab = pd.read_csv(latest_res)
    print(f"Cargando resultados desde: {latest_res}")
    
    # Mostrar métricas primarias en escenario standard
    std_df = df_colab[df_colab['scenario'] == 'standard'] if 'scenario' in df_colab.columns else df_colab
    primary_cols = [c for c in ['seed', 'variant', 'ndcg_at_10_retrieval_solo', 'ndcg_at_10_two_stage_N50', 'cand_recall_cut0_at_200', 'cand_recall_cut1_5_at_200'] if c in std_df.columns]
    
    print("\n" + "=" * 75)
    print("RESULTADOS PRIMARIOS — ESCENARIO ESTÁNDAR (COLAB)")
    print("=" * 75)
    display(std_df[primary_cols].round(4))
    
    # Escenario en frío
    if 'scenario' in df_colab.columns and 'cold_start' in df_colab['scenario'].values:
        cold_df = df_colab[df_colab['scenario'] == 'cold_start']
        cold_cols = [c for c in ['seed', 'variant', 'cand_recall_cold_at_200', 'cand_recall_cold_at_50', 'cand_recall_cold_at_10'] if c in cold_df.columns]
        print("\n" + "=" * 75)
        print("RESULTADOS PRIMARIOS — ESCENARIO COLD-START (COLAB)")
        print("=" * 75)
        display(cold_df[cold_cols].round(4))
        
    # Contrastes pareados si hay más de 1 semilla
    if len(SEEDS) > 1 and 'ndcg_at_10_retrieval_solo' in std_df.columns:
        print("\n" + "=" * 75)
        print("CONTRASTES PAREADOS EN COLAB")
        print("=" * 75)
        piv = std_df.pivot(index='seed', columns='variant', values='ndcg_at_10_retrieval_solo')
        for c_pair in [('C2', 'D2'), ('D2', 'A'), ('B', 'A')]:
            v1, v2 = c_pair
            if v1 in piv.columns and v2 in piv.columns:
                diff = piv[v1] - piv[v2]
                print(f"{v1} - {v2} (Retrieval NDCG@10): Media = {diff.mean():+.4f} ± {diff.std():.4f} | Positivas = {(diff > 0).sum()}/{len(diff)}")
else:
    print(f"No se encontraron archivos de resultados en {target_results_dir}.")

--- 
## 7. Tabla Consolidada de Referencia (10 Semillas en GPU RTX 4060 Local)
A continuación se presentan los resultados oficiales consolidados a lo largo de las 10 semillas (42 a 51) ejecutadas en el entorno local del autor, mostrando la media y el desvío estándar por variante, escenario y métrica.

In [ ]:
ref_file = "results/gpu/stage4/stage4_consolidated_summary_10seeds.csv"
if os.path.exists(ref_file):
    df_ref = pd.read_csv(ref_file)
    print("=" * 95)
    print("TABLA DE REFERENCIA OFICIAL — 10 SEMILLAS EN GPU RTX 4060 LOCAL (PyTorch 2.6.0+cu124)")
    print("=" * 95)
    
    # Mostrar escenario estándar y cold_start formateados
    for scen in ['standard', 'cold_start']:
        sub = df_ref[df_ref['scenario'] == scen]
        if not sub.empty:
            print(f"\n>>> Escenario: {scen.upper()} (10 semillas — Media ± Desvío Estándar)")
            display_df = pd.DataFrame({'variant': sub['variant']})
            if scen == 'standard':
                display_df['Retrieval NDCG@10'] = sub['ndcg_at_10_retrieval_solo_mean'].map(lambda m: f"{m:.4f}") + " ± " + sub['ndcg_at_10_retrieval_solo_std'].map(lambda s: f"{s:.4f}")
                display_df['Two-Stage N=50 NDCG@10'] = sub['ndcg_at_10_two_stage_N50_mean'].map(lambda m: f"{m:.4f}") + " ± " + sub['ndcg_at_10_two_stage_N50_std'].map(lambda s: f"{s:.4f}")
                display_df['CandRecall Cut 0'] = sub['cand_recall_cut0_at_200_mean'].map(lambda m: f"{m*100:.2f}%") + " ± " + sub['cand_recall_cut0_at_200_std'].map(lambda s: f"{s*100:.2f}%" if pd.notna(s) else "0.00%")
                display_df['CandRecall Cut 1-5'] = sub['cand_recall_cut1_5_at_200_mean'].map(lambda m: f"{m*100:.2f}%") + " ± " + sub['cand_recall_cut1_5_at_200_std'].map(lambda s: f"{s*100:.2f}%")
                display_df['Cobertura Retr.'] = sub['catalog_coverage_retrieval_solo_mean'].map(lambda m: f"{m*100:.2f}%") + " ± " + sub['catalog_coverage_retrieval_solo_std'].map(lambda s: f"{s*100:.2f}%")
                display_df['Pop. Media Retr.'] = sub['mean_pop_retrieval_solo_mean'].map(lambda m: f"{m:.2f}") + " ± " + sub['mean_pop_retrieval_solo_std'].map(lambda s: f"{s:.2f}")
            else:
                display_df['CandRecall Cold @200'] = sub['cand_recall_cold_at_200_mean'].map(lambda m: f"{m*100:.2f}%") + " ± " + sub['cand_recall_cold_at_200_std'].map(lambda s: f"{s*100:.2f}%")
                display_df['CandRecall Cold @50'] = sub['cand_recall_cold_at_50_mean'].map(lambda m: f"{m*100:.2f}%") + " ± " + sub['cand_recall_cold_at_50_std'].map(lambda s: f"{s*100:.2f}%")
                display_df['CandRecall Cold @10'] = sub['cand_recall_cold_at_10_mean'].map(lambda m: f"{m*100:.2f}%") + " ± " + sub['cand_recall_cold_at_10_std'].map(lambda s: f"{s*100:.2f}%")
                display_df['Retrieval NDCG@10'] = sub['ndcg_at_10_retrieval_solo_mean'].map(lambda m: f"{m:.4f}") + " ± " + sub['ndcg_at_10_retrieval_solo_std'].map(lambda s: f"{s:.4f}")
                display_df['Two-Stage N=50 NDCG@10'] = sub['ndcg_at_10_two_stage_N50_mean'].map(lambda m: f"{m:.4f}") + " ± " + sub['ndcg_at_10_two_stage_N50_std'].map(lambda s: f"{s:.4f}")
            display(display_df)
else:
    print(f"Archivo de referencia {ref_file} no encontrado.")